In [1]:
import os
import re

In [2]:
def rename_LivDet2013(root_directory):
    # Bảng ánh xạ ngón tay chuẩn (0-9)
    finger_map = {
        'Rthb': '0', 'Ridx': '1', 'Rmdl': '2', 'Rrng': '3', 'Rltl': '4',
        'Lthb': '5', 'Lidx': '6', 'Lmdl': '7', 'Lrng': '8', 'Lltl': '9'
    }
    
    for root, dirs, files in os.walk(root_directory):
        # Sắp xếp danh sách file để thứ tự đánh số (0, 1, 2...) luôn cố định
        files.sort()
        
        # Bộ đếm riêng cho từng folder và từng ngón tay của từng User
        # Key: "UserID_FingerCode", Value: số thứ tự hiện tại
        folder_counters = {}

        for filename in files:
            if not filename.endswith(".png"):
                continue
            
            # 1. Trích xuất ID người dùng
            user_id_match = re.match(r'^(\d+)', filename)
            user_id = user_id_match.group(1) if user_id_match else "Unknown"
            
            # 2. Xác định ngón tay
            finger_code = "U"
            for key, value in finger_map.items():
                if key in filename:
                    finger_code = value
                    break
            
            if finger_code == "U":
                continue

            # 3. Tạo Key để quản lý thứ tự
            # Ví dụ: "072_6" (User 072, Ngón 6)
            key = f"{user_id}_{finger_code}"
            
            # 4. Lấy số thứ tự hiện tại và tăng lên 1
            current_idx = folder_counters.get(key, 0)
            folder_counters[key] = current_idx + 1
            
            # 5. Tạo tên file mới: ID_Ngón_ThứTự.png
            new_name = f"{user_id}_{finger_code}_{current_idx}.png"
            
            old_path = os.path.join(root, filename)
            new_path = os.path.join(root, new_name)

            # Kiểm tra tránh trùng lặp nếu bạn chạy script nhiều lần
            if filename == new_name:
                continue

            # print(f"Renaming: {filename} -> {new_name}")
            
            # Thực thi đổi tên
            try:
                os.rename(old_path, new_path)
            except OSError as e:
                print(f"[LỖI] Không thể đổi tên {filename}: {e}")

    print("--- Hoàn tất quá trình xử lý! ---")

In [6]:
# Đường dẫn của bạn
path = "./data/LivDet/livdet2013/Italdata/Test/Live"
rename_LivDet2013(path)

--- Hoàn tất quá trình xử lý! ---


In [7]:
def rename_LivDet2011(root_directory):    
    for root, dirs, files in os.walk(root_directory):
        # Sắp xếp file để đảm bảo thứ tự đánh số luôn cố định mỗi lần chạy
        files.sort()
        
        # Reset bộ đếm cho mỗi thư mục con
        # Key: "UserID_Finger", Value: số thứ tự hiện tại (0, 1, 2...)
        folder_counters = {}
        
        for filename in files:
            if not filename.lower().endswith(('.bmp', '.png')):
                continue
            
            # Tách phần tên và đuôi file
            name_part, ext = os.path.splitext(filename)
            parts = name_part.split('_')
            
            # 1. User ID (phần tử đầu tiên)
            user_id = parts[0]
            
            # 2. Tìm vị trí ngón tay (R1-5 hoặc L1-5)
            finger_found = None
            for part in parts:
                if re.match(r'^[RL][1-5]$', part):
                    finger_found = part
                    break

            # 3. Tiến hành xử lý số thứ tự và đổi tên
            if finger_found:
                # Tạo key duy nhất cho mỗi ngón tay của mỗi người dùng
                key = f"{user_id}_{finger_found}"
                
                # Lấy số thứ tự hiện tại từ bộ đếm và tăng lên 1
                current_idx = folder_counters.get(key, 0)
                folder_counters[key] = current_idx + 1
                
                # Tên mới tối giản: ID_Finger_ThứTự.ext
                # Sample ID gốc trong tên file sẽ bị loại bỏ và thay bằng current_idx
                new_name = f"{user_id}_{finger_found}_{current_idx}{ext}"
                
                old_path = os.path.join(root, filename)
                new_path = os.path.join(root, new_name)
                
                # Nếu tên đã trùng khớp thì bỏ qua (tránh báo lỗi rename cùng tên)
                if filename == new_name:
                    continue

                # print(f"Renaming: {filename} -> {new_name}")
                
                try:
                    # LƯU Ý: Trên Linux, lệnh này sẽ GHI ĐÈ nếu new_path đã tồn tại
                    os.rename(old_path, new_path)
                except Exception as e:
                    print(f"[LỖI] Không thể đổi tên {filename}: {e}")
            else:
                print(f"[BỎ QUA] Không tìm thấy ký hiệu R/L: {filename}")

    print("--- Hoàn tất quá trình đánh số thứ tự! ---")

In [11]:
# Đường dẫn đến folder chứa các file .bmp của bạn
path = "./data/LivDet/livdet2011/Sagem/Test/Live"
rename_LivDet2011(path)

--- Hoàn tất quá trình đánh số thứ tự! ---
